# Streamlit Homework: Sales Dashboard

## Overview

In this homework you'll build a complete interactive sales dashboard using Streamlit. You'll combine everything from Session 14 — display elements, widgets, layout, and Plotly charts — into a single working app.

**What you'll build:** A dashboard that lets users filter e-commerce sales data and explore it through KPI metrics, charts, and an interactive data table with CSV export.

## Estimated Time: 2-3 hours

## Files

| File | Description |
|------|-------------|
| `dashboard.py` | **Your file** — skeleton with TODO comments. Implement each section. |
| `dashboard_solved.py` | Complete solution (don't peek until you've tried!) |
| `data/sales_dashboard.csv` | Dataset: 200 e-commerce orders |
| `deploy/` | Files for optional deployment to Streamlit Community Cloud |

## How to Work on This Homework

1. Open `dashboard.py` in VS Code
2. Run it: `streamlit run dashboard.py`
3. Implement each TODO section (1 through 4)
4. After each change, save the file and click **Rerun** in the browser
5. When done, compare your result with `streamlit run dashboard_solved.py`

---

## Dataset

The dataset (`data/sales_dashboard.csv`) contains 200 e-commerce orders with these columns:

| Column | Type | Example |
|--------|------|---------|
| `order_id` | str | `ORD001` |
| `order_date` | str | `2025-06-15` |
| `product_name` | str | `Wireless Mouse` |
| `category` | str | `Electronics`, `Office`, `Software`, `Accessories` |
| `quantity` | int | `3` |
| `unit_price` | float | `29.99` |
| `region` | str | `North America`, `Europe`, `Asia Pacific`, `Latin America` |
| `sales_rep` | str | `Alice Johnson` |
| `status` | str | `Completed`, `Pending`, `Cancelled` |

Revenue is computed in-app as `quantity * unit_price`.

---

## What's Already Done For You

The `dashboard.py` file already has:

- All imports (`streamlit`, `pandas`, `plotly.express`, `pathlib`)
- Page configuration (`st.set_page_config`)
- Data loading with `@st.cache_data` (loads CSV, converts dates, computes revenue)
- The page title

You need to implement **4 TODO sections**.

---

## TODO 1: Sidebar Filters

Create a sidebar with 4 filters:

### Date Range

Use `st.sidebar.date_input` for a start date and an end date.

```python
min_date = df["order_date"].min().date()
max_date = df["order_date"].max().date()

start_date = st.sidebar.date_input("Start date", value=min_date, min_value=min_date, max_value=max_date)
end_date = st.sidebar.date_input("End date", value=max_date, min_value=min_date, max_value=max_date)
```

### Category, Region, and Status

Use `st.sidebar.multiselect` for each. All options should be selected by default.

```python
selected_categories = st.sidebar.multiselect(
    "Categories:",
    options=df["category"].unique().tolist(),
    default=df["category"].unique().tolist(),
)
```

Do the same for regions and statuses.

---

## TODO 2: Apply Filters

Combine all filter conditions with `&` to create a filtered DataFrame:

```python
filtered = df[
    (df["order_date"].dt.date >= start_date)
    & (df["order_date"].dt.date <= end_date)
    & (df["category"].isin(selected_categories))
    & (df["region"].isin(selected_regions))
    & (df["status"].isin(selected_statuses))
]
```

**Optional:** Add a caption showing how many rows match:
```python
st.caption(f"Showing {len(filtered)} of {len(df)} orders")
```

---

## TODO 3: KPI Metrics Row

Create 4 columns and display key metrics:

| Column | Metric | Format |
|--------|--------|--------|
| 1 | Total Revenue | `$X,XXX.XX` |
| 2 | Total Orders | count of rows |
| 3 | Average Order Value | `$X,XXX.XX` |
| 4 | Top Category | category with highest total revenue |

**Hints:**
- Use `st.columns(4)` to create the layout
- Use `col.metric("Label", "Value")` to display each metric
- Format numbers with f-strings: `f"${value:,.2f}"`
- Find the top category: `filtered.groupby("category")["revenue"].sum().idxmax()`
- Handle the empty DataFrame case: check `if len(filtered) > 0` before computing averages or max values

---

## TODO 4: Visualization Tabs

Create 4 tabs using `st.tabs(["Overview", "By Category", "By Region", "Data"])`.

### Overview Tab — Monthly Revenue Line Chart

Group by month and sum revenue, then display as a line chart:

```python
monthly = (
    filtered.groupby(filtered["order_date"].dt.to_period("M"))["revenue"]
    .sum()
    .reset_index()
)
monthly["order_date"] = monthly["order_date"].astype(str)

fig = px.line(monthly, x="order_date", y="revenue", markers=True)
st.plotly_chart(fig, use_container_width=True)
```

### By Category Tab — Horizontal Bar Chart

Group by category, sort by revenue, and display as a horizontal bar chart:

```python
fig = px.bar(category_df, x="revenue", y="category", orientation="h", color="category")
```

### By Region Tab — Pie Chart

Group by region and display as a pie chart:

```python
fig = px.pie(region_df, values="revenue", names="region")
```

### Data Tab — Table + Download

Display the filtered DataFrame and add a download button:

```python
st.dataframe(filtered, use_container_width=True)

csv_data = filtered.to_csv(index=False)
st.download_button(
    label="Download as CSV",
    data=csv_data,
    file_name="filtered_sales_data.csv",
    mime="text/csv",
)
```

**For all tabs:** Show `st.info("No data to display.")` when the filtered DataFrame is empty.

---

## Bonus: Deploy Your Dashboard

Once your dashboard works locally, you can optionally deploy it to the web for free.

### Steps

1. Create a new **public** GitHub repository
2. Copy these files into it:
   - `dashboard.py` (your completed version)
   - `data/sales_dashboard.csv`
   - `deploy/requirements.txt` (rename to `requirements.txt` in the root)
   - `deploy/.streamlit/config.toml` (optional, for theming)
3. Push to GitHub
4. Go to [share.streamlit.io](https://share.streamlit.io) and deploy

See `deploy/` folder for the required files and refer to Session 14's deployment section for detailed instructions.

---

## Solution

When you're done (or stuck), run the solution:

```bash
streamlit run dashboard_solved.py
```